In [1]:
import pandas as pd
import scanpy as sc
import numpy as np
import anndata as ad



In [2]:
filename = "TCGA-BRCA.star_counts"
df_original = pd.read_csv(f'../data/{filename}.tsv', delimiter='\t', index_col=False)

# for MLP

In [3]:
df = df_original.rename(columns={'Ensembl_ID': 'Unnamed: 0-1'})
df.columns = df.columns.str.split("-").str[:-1].str.join("-")

df = df.T
df.columns = df.iloc[0]   # pierwszy wiersz → nazwy kolumn
df = df.iloc[1:]


In [4]:
df.columns = df.columns.str.split(".").str[0]

features = pd.read_csv("../data/gene_info.csv")
features = features[["feature_id", "feature_name"]]

id_to_symbol = dict(
    zip(features["feature_id"], features["feature_name"])
)

df = df.rename(columns=id_to_symbol)

df = df.loc[:, df.columns.notnull()]
df = df.loc[:, ~df.columns.duplicated()]

df.shape

(1226, 60616)

In [5]:
df.head()

Unnamed: 0,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,RP11-357C3.5,RP11-73F12.1,RP11-680A11.7,RP11-439H13.3,RP11-73F12.2,CTD-3214H19.17,RP11-122G18.14,AC006486.14,RP5-1087E8.6,PANO1
TCGA-D8-A146,11.73767,7.721099,11.042343,11.03686,9.131857,9.136991,11.869594,11.004923,11.345405,11.671541,...,0.0,0.0,5.321928,0.0,0.0,0.0,8.184875,0.0,2.807355,4.954196
TCGA-AQ-A0Y5,9.78136,3.321928,11.357552,10.754888,8.721099,7.774787,11.902752,12.030322,11.074141,11.624339,...,0.0,0.0,3.906891,0.0,0.0,0.0,9.584963,0.0,2.807355,5.247928
TCGA-C8-A274,13.122504,0.0,11.506308,12.21826,10.973697,7.954196,10.239599,11.017504,10.817783,12.313166,...,0.0,0.0,5.392317,0.0,0.0,0.0,9.942515,0.0,3.906891,4.321928
TCGA-BH-A0BD,11.016808,6.686501,10.801708,11.190442,10.761551,8.810572,11.800091,11.179287,10.715104,11.683433,...,0.0,0.0,4.392317,0.0,1.0,0.0,9.590587,0.0,3.0,3.807355
TCGA-B6-A1KC,11.0,3.807355,11.074141,10.857981,9.550747,7.294621,11.249113,11.325868,10.83289,12.506803,...,0.0,0.0,5.672425,0.0,0.0,0.0,10.625709,0.0,2.321928,3.807355


In [6]:
# keep only genes that were originally in lung data
smaple_data_path = '../data/Kim2020_Lung.h5ad'
adata = sc.read_h5ad(smaple_data_path)
columns = adata.var['gene_name']
gene_list = columns.dropna().unique().tolist()
df_filtered = df[ df.columns.intersection(gene_list) ]


In [7]:
df_filtered = df_filtered[~df_filtered.index.duplicated(keep="first")]


In [8]:
# save to file
df_filtered.to_csv(f"../data/{filename}.csv")


In [9]:
df1 = pd.read_csv(f"../data/{filename}.csv")

# for scGPT

In [10]:
# create anndata df

X = df_filtered.values.astype(np.float32)

obs = pd.DataFrame(index=df_filtered.index)
obs["sample"] = df_filtered.index

var = pd.DataFrame(index=df_filtered.columns)
var["gene_name"] = df_filtered.columns

adata = ad.AnnData(
    X=X,
    obs=obs,
    var=var
)

adata

AnnData object with n_obs × n_vars = 1095 × 20560
    obs: 'sample'
    var: 'gene_name'

In [11]:
adata.write(f"../data/adata_{filename}.h5ad")
